# Agent evaluation with Opik: a 20-minute quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/comet-ml/opik-examples/blob/main/guides/end_to_end_agent_evaluation/agent_evaluation_quickstart.ipynb)

A short, hands-on quickstart for evaluating an LLM agent with
[Opik](https://github.com/comet-ml/opik). We trace a small support agent, define a
judge once, run an experiment, and align the judge against human labels. It is a
trimmed version of the full guide, **end-to-end agent evaluation with Opik**, which
adds the release gate, online evaluation on production traffic, guardrails, and
closing the loop.

What we'll do: **observe** the agent, **define a judge once**, **run an
experiment**, and **align the judge with humans**.

## Setup

Install Opik and an LLM client, configure Opik, and define the sample agent.

In [ ]:
%pip install --quiet --upgrade opik anthropic matplotlib ipywidgets

In [ ]:
import os
import opik

OPIK_PROJECT_NAME = "end-to-end-agent-eval"

# --- Self-hosted? point at your install (or set these as env vars) ---
# os.environ["OPIK_URL_OVERRIDE"] = "https://opik.my-company.internal/api"

# Reads OPIK_API_KEY / OPIK_WORKSPACE from the environment; prompts if not set.
opik.configure(project_name=OPIK_PROJECT_NAME)

In [ ]:
import getpass

AGENT_MODEL = "claude-haiku-4-5-20251001"   # the agent's model (Anthropic SDK)
# The judges take a LiteLLM "provider/model" string, so they can use any provider.
# Use a stronger model than the agent's for judging in practice.
JUDGE_MODEL = os.environ.get("OPIK_EXAMPLES_MODEL", f"anthropic/{AGENT_MODEL}")

for key_var in dict.fromkeys(["ANTHROPIC_API_KEY", f"{JUDGE_MODEL.split('/')[0].upper()}_API_KEY"]):
    if not os.environ.get(key_var):
        os.environ[key_var] = getpass.getpass(f"{key_var}: ")

### The system prompt as a tracked artifact

The instruction that governs the agent is not a throwaway string; it is a
controlled artifact. Register it in the **Opik prompt library** as a versioned
prompt via `client.create_prompt`; every edit creates a new commit.

Then **link that version to each trace** with
`opik_context.update_current_trace(prompts=[SYSTEM_PROMPT])` (and to each
experiment with `evaluate(prompts=[...])`, in the experiment below). That is the real linking
mechanism: Opik records the version under the trace's `opik_prompts`, and the UI
shows the prompt on the trace. The result is a full audit trail, which prompt
version produced this answer and which version cleared the gate. Establish this
from the first trace. You can also create and edit prompts directly in the Opik UI.

Docs: [Prompt management](https://www.comet.com/docs/opik/prompt_engineering/prompt_management)

In [ ]:
client = opik.Opik()

SYSTEM_PROMPT = client.create_prompt(
    name="billing-support-system",
    prompt=("You are Acme Cloud billing support. Answer ONLY from the provided "
            "context. If unknown, say so."),
    description="System prompt governing the Acme Cloud billing support agent.",
)
print("system prompt version (commit):", SYSTEM_PROMPT.commit)

### The sample agent: billing support with RAG + a tool

Our agent answers billing questions for a fictional product, **Acme Cloud**. It
can:

- **`search_docs`**: retrieve policy snippets (a tiny keyword "retriever" so the
  notebook has no vector-DB dependency),
- **`get_order_status`**: look up an order by id (a second tool, so the agent has
  a real *tool-selection* decision to make),
- **`llm_answer`**: compose a grounded answer from the retrieved context.

Each function is decorated with `@opik.track`. The `type=` argument marks a span
as a `tool` or `llm` span; that is what later lets us score *individual spans*,
not just the final answer.

In [ ]:
import re
import opik
from opik import opik_context
from anthropic import Anthropic
from opik.integrations.anthropic import track_anthropic

claude = track_anthropic(Anthropic())   # wrapping the client auto-captures tokens, cost & latency

# --- tiny "knowledge base" -------------------------------------------------
DOCS = {
    "refunds": "Acme Cloud refunds are available within 30 days of purchase. "
               "Annual plans are refunded pro-rata. Refunds take 5-7 business days.",
    "cancellation": "You can cancel any time from Billing > Subscription. "
                    "Cancelling stops the next renewal; access continues until period end.",
    "trials": "The free trial lasts 14 days. No card is charged until the trial ends.",
    "invoices": "Invoices are emailed monthly and downloadable under Billing > Invoices.",
}
ORDERS = {"A-1001": "shipped", "A-1002": "processing", "A-1003": "refunded"}


@opik.track(type="tool")
def search_docs(query: str) -> list:
    """Keyword retriever: returns matching policy snippets."""
    q = query.lower()
    hits = [text for key, text in DOCS.items()
            if key in q or any(w in q for w in key.split())]
    # fall back to a couple of docs so the LLM always has something to ground on
    return hits or list(DOCS.values())[:2]


@opik.track(type="tool")
def get_order_status(order_id: str) -> str:
    return ORDERS.get(order_id.upper(), "unknown order id")


@opik.track(type="llm")
def llm_answer(question: str, context: list) -> str:
    ctx = "\n".join(f"- {c}" for c in context)
    resp = claude.messages.create(
        model=AGENT_MODEL,
        max_tokens=512,
        system=SYSTEM_PROMPT.prompt,
        messages=[{"role": "user", "content": f"Context:\n{ctx}\n\nQuestion: {question}"}],
    )
    return resp.content[0].text


@opik.track(project_name=OPIK_PROJECT_NAME)
def support_agent(question: str) -> dict:
    """Root trace. Picks tools, retrieves context, answers."""
    # link the prompt version to this trace (shows under the trace's prompt in Opik)
    opik_context.update_current_trace(prompts=[SYSTEM_PROMPT])

    order_ids = re.findall(r"[A-Za-z]-\d{4}", question)
    if order_ids:
        get_order_status(order_ids[0])

    context = search_docs(question)
    answer = llm_answer(question, context)
    return {"output": answer, "context": context}

## Instrument and observe

You cannot evaluate what you cannot see. Run the agent once; Opik records a
**trace** with a nested **span** per decorated call (two tool spans and one LLM
span), capturing inputs and outputs and, for the LLM span, token usage, cost and
latency automatically.

Docs: [Tracing / log traces](https://www.comet.com/docs/opik/tracing/log_traces)

In [ ]:
result = support_agent("How do refunds work, and what's the status of order A-1001?")
opik.flush_tracker()

print(result["output"])
print("\nOpen the Opik UI to see the trace: a root span with get_order_status and "
      "search_docs (tool) and llm_answer (llm) nested underneath, plus cost, latency, "
      "and the linked prompt version. That nested span tree IS the agent's trajectory; "
      "you read it in Opik rather than hand-building it.")

## Define a judge once

A **judge** scores a quality a heuristic cannot: here, is the answer consistent with
Acme's billing policy and free of invented facts? We keep the rubric in a constant so
the *same* definition can be reused in an experiment now and as an online rule later.

Docs: [G-Eval](https://www.comet.com/docs/opik/evaluation/metrics/g_eval)

In [ ]:
from opik.evaluation.metrics import GEval

GROUNDING_TASK_INTRO = (
    "You are a strict reviewer of an Acme Cloud billing-support answer."
)
GROUNDING_CRITERIA = (
    "Return 1 only if the answer is factually consistent with Acme policy "
    "(refunds within 30 days and taking 5-7 business days; a 14-day free trial; "
    "cancel anytime with access until period end; invoices under Billing > "
    "Invoices) AND it invents no amounts, dates, or promises not in that policy. "
    "If the question is outside this policy, the answer must decline rather than "
    "guess. Otherwise return 0."
)

grounding_judge = GEval(
    task_introduction=GROUNDING_TASK_INTRO,
    evaluation_criteria=GROUNDING_CRITERIA,
    model=JUDGE_MODEL,
    name="grounding",
)

out = support_agent("How long do I have to get a refund?")
res = grounding_judge.score(output=out["output"])
print("grounding:", res.value, "-", res.reason)

## Run an experiment

An **experiment** runs the agent over a small golden dataset and scores every item.
We link the prompt version to the run, so the result is auditable: which prompt
produced these scores.

Docs: [Evaluate your LLM](https://www.comet.com/docs/opik/evaluation/evaluate_your_llm)

In [ ]:
from opik.evaluation import evaluate
from opik.evaluation.metrics import AnswerRelevance, Hallucination

dataset = client.get_or_create_dataset(
    name="support-agent-golden-lite",
    description="A few billing Q&A cases for the quickstart.",
)
dataset.insert([
    {"input": "How long do I have to get a refund?",
     "expected_output": "Within 30 days of purchase.", "context": [DOCS["refunds"]]},
    {"input": "How long is the free trial?",
     "expected_output": "14 days.", "context": [DOCS["trials"]]},
    {"input": "How do I cancel my subscription?",
     "expected_output": "From Billing > Subscription; access lasts until period end.",
     "context": [DOCS["cancellation"]]},
])

def eval_task(item):
    out = support_agent(item["input"])
    return {"output": out["output"], "context": out["context"]}

baseline = evaluate(
    dataset=dataset,
    task=eval_task,
    scoring_metrics=[AnswerRelevance(model=JUDGE_MODEL),
                     Hallucination(model=JUDGE_MODEL),
                     grounding_judge],      # the SAME judge object from above
    experiment_name="support-agent-lite-baseline",
    experiment_config={"model": AGENT_MODEL, "version": "baseline"},
    prompts=[SYSTEM_PROMPT],                # link the prompt version to the experiment
    verbose=0,
)
print("Experiment created; open it in Opik for per-item and aggregate scores.")

## Align the judge with humans

So far we used the judge to **score the agent**. This section does a different
job: it **validates the judge itself**. An LLM judge is a model too, so before you
trust it to gate a release (or feed an audit record), you align it against **human
labels**. In Opik you do this with **annotation queues** + **feedback scores**: a
repeatable workflow that you iterate until the judge agrees with your reviewers.

Keep the two jobs distinct:

1. **Score the agent** (the experiment above): is the agent good enough to ship?
2. **Align the judge** (this section): is the judge trustworthy enough to make that call?

> This section keeps alignment **light**: the annotation-queue mechanics plus the
> workflow. For a full calibration study, see the judge alignment docs linked at the
> end.

Docs: [Annotation queues](https://www.comet.com/docs/opik/evaluation/annotation_queues)

### The alignment workflow (what you are actually doing)

1. **Sample** a representative set of production (or dataset) traces.
2. **Label them by hand** against a clear rubric: **binary pass/fail plus a
   written critique**, not a 1-5 Likert; graded scales inflate disagreement.
3. **Compare judge vs human** as **TPR and TNR** (true-positive / true-negative
   rate), never raw agreement %: a judge that passes everything looks 90% accurate
   on a mostly-good sample while catching nothing.
4. **Iterate the judge prompt** until agreement converges (teams often reach
   over 90% in a few iterations).
5. **Benchmark against the human-to-human ceiling** (~70-75% is typical); your judge
   cannot be more consistent than your humans are with each other.

The mechanics below create a binary feedback definition, build an annotation
queue with a rubric, and route a sample of traces to it. Humans label in the Opik
UI; you then read the labels back and do a light agreement read against the judge.

> **SDK or UI.** Everything in the next cell can also be done in the Opik web app:
> feedback definitions under **Configuration > Feedback definitions**, and queues
> under the **Annotation Queues** tab. The SDK version below is the same thing made
> repeatable, so you can wire it into a pipeline.

In [ ]:
from opik.rest_api.types import FeedbackCreate_Categorical, CategoricalFeedbackDetailCreate

# 1) A binary feedback definition (pass/fail). Comments on the queue carry the
#    written critique that pairs with each label.
def ensure_binary_feedback(name: str):
    try:
        client.rest_client.feedback_definitions.create_feedback_definition(
            request=FeedbackCreate_Categorical(
                name=name,
                details=CategoricalFeedbackDetailCreate(categories={"fail": 0.0, "pass": 1.0}),
            )
        )
        print("created feedback definition:", name)
    except Exception as e:
        print(f"feedback definition '{name}' already exists or not creatable here:", type(e).__name__)

ensure_binary_feedback("human_grounding")

# 2) An annotation queue with a clear rubric, comments enabled for the critique.
queue = client.create_traces_annotation_queue(
    name="Grounding alignment review",
    project_name=OPIK_PROJECT_NAME,
    instructions=("Mark 'pass' only if the answer is consistent with Acme policy "
                  "and invents nothing. Add a one-line critique in the comment."),
    comments_enabled=True,
    feedback_definition_names=["human_grounding"],
)

# 3) Route the RIGHT traces in. This is JUDGE calibration, not agent calibration,
#    so we annotate exactly the traces this experiment scored, nothing else. The
#    experiment result carries their ids directly, so the selection is bounded to
#    this run no matter how many other traces the project accumulates.
trace_ids = [tr.test_case.trace_id for tr in baseline.test_results]
exp_traces = [client.get_trace_content(tid) for tid in trace_ids]
queue.add_traces(exp_traces)
print(f"Queued {len(exp_traces)} traces from the baseline experiment for human review.")

In [ ]:
# Read human labels back and compare the judge to them, over EXACTLY the experiment
# traces we queued. The judge is continuous and the humans are binary, so we binarise
# the judge at the SAME threshold a release gate would use; alignment is measured at
# the operating point you ship. (Run this AFTER reviewers have labelled in the UI.)
OPERATING_THRESHOLD = 0.7   # the grounding bar the full guide's release gate uses

def confusion(trace_ids, human="human_grounding", judge="grounding",
              threshold=OPERATING_THRESHOLD):
    """Judge-vs-human confusion matrix + TPR/TNR over the given experiment traces."""
    tp = tn = fp = fn = 0
    for tid in trace_ids:
        fs = {s.name: s.value for s in (client.get_trace_content(tid).feedback_scores or [])}
        if human in fs and judge in fs:
            h = fs[human] >= 0.5            # human pass/fail (already binary)
            j = fs[judge] >= threshold      # judge binarised at the gate threshold
            tp += (h and j); tn += ((not h) and (not j))
            fp += ((not h) and j); fn += (h and (not j))
    tpr = tp / (tp + fn) if (tp + fn) else None   # of human-PASS, judge also passed
    tnr = tn / (tn + fp) if (tn + fp) else None   # of human-FAIL, judge also failed
    return {"tp": tp, "tn": tn, "fp": fp, "fn": fn,
            "n": tp + tn + fp + fn, "TPR": tpr, "TNR": tnr}

cm = confusion([tr.test_case.trace_id for tr in baseline.test_results])
print(f"judge vs human at threshold {OPERATING_THRESHOLD}:", cm)
print("(Populates once humans have labelled in the UI.)")

### Reading TPR and TNR

- **TPR** (of the answers *humans passed*, the fraction the judge also passed): low
  TPR means the judge is **too strict**; it fails good answers and blocks releases it
  should not.
- **TNR** (of the answers *humans failed*, the fraction the judge also failed): **this
  is the dangerous one.** Low TNR means the judge **passes bad answers**, so it waves
  regressions through and pollutes your audit record.
- Read them separately, never as a single accuracy number; on a mostly-good sample a
  do-nothing judge scores high accuracy while catching nothing. Iterate the judge
  prompt until both clear the bar your risk tier needs, remembering the human-to-human
  ceiling caps how high either can go.

## Where to go next

That is the core loop: observe, define a judge once, run an experiment, align the
judge. The **full guide** adds the release gate, online evaluation on production
traffic (the same judge as an automation rule), guardrails, and closing the loop by
promoting production failures back into the dataset.

- Full guide: `end_to_end_agent_evaluation.ipynb`
- Metrics reference: https://www.comet.com/docs/opik/evaluation/metrics/overview
- Aligning judges with human scores: https://www.comet.com/docs/opik/evaluation/judge_alignment
- Online evaluation rules: https://www.comet.com/docs/opik/production/online-evaluation/rules